In [158]:
import os
import numpy as np
import matplotlib.pyplot as plt
from os.path import join
from PIL import Image
from glob import glob
import random

In [159]:
nnUNet_raw = "data/nnUNet_raw"
nnUNet_preprocessed = "data/nUNet_preprocessed"

os.environ["nnUNet_raw"] = nnUNet_raw
os.environ["nnUNet_preprocessed"] = nnUNet_preprocessed

In [160]:
def get_dataset_name(dataset_id):
    dataset_id = str(dataset_id).zfill(3)
    dataset_name = [
        d for d in os.listdir(nnUNet_raw)
        if d.startswith(f"Dataset{dataset_id}") and os.path.isdir(join(nnUNet_raw, d))
    ]
    if len(dataset_name) != 1:
        raise RuntimeError(f"Found {len(dataset_name)} datasets with id {dataset_id}, expected 1")
    return dataset_name[0]

# Task Difficulty Index

## 1. Target Geometry

### a) Target variability

In [558]:
import numpy as np
from tqdm import tqdm

def compute_target_stats(mask_paths):
    """
    masks: iterable of 2D/3D binary arrays (0/1 or bool), all same shape per sample
    Returns: dict with mean, std, CV, median, IQR of area fraction.
    """
    fracs = []
    for m_path in tqdm(mask_paths):
        m = np.array(Image.open(m_path))
        m = m.astype(bool)
        fracs.append(m.mean())  # area / image_area
    fracs = np.asarray(fracs, dtype=float)

    mean = fracs.mean()
    std  = fracs.std(ddof=1) if len(fracs) > 1 else 0.0
    cv   = std / (mean + 1e-12)

    q25, q50, q75 = np.quantile(fracs, [0.25, 0.50, 0.75])
    return {
        "n": int(len(fracs)),
        "mean_area_frac": float(mean),
        "std_area_frac": float(std),
        "cv_area_frac": float(cv),
        "median_area_frac": float(q50),
        "iqr_area_frac": float(q75 - q25),
        "area_fracs": fracs,  # keep if you want hist/violin
    }


In [577]:
dataset_ids = [300, 301, 302, 304, 305]

target_var = {}
for dataset_id in dataset_ids:
    dataset_name = get_dataset_name(dataset_id)

    targets = glob(join(nnUNet_raw, dataset_name, "labelsTr", "*.png"))
    # num_cases = 100
    # targets = targets[:num_cases]
    target_stats = compute_target_stats(targets)
    target_var[dataset_name] = target_stats['cv_area_frac']

# Print results sorted by value
print("\nResults")
for dataset_name, cv_val in sorted(target_var.items(), key=lambda x: x[1]):
    print(f"{dataset_name:<28}: {cv_val:.4f}")

100%|██████████| 2211/2211 [00:01<00:00, 1836.44it/s]


Results
Dataset302_EchoNet-Dynamic  : 0.3504
Dataset301_busbra           : 0.8913
Dataset300_isic2018         : 0.9652
Dataset304_BUSI             : 1.2433
Dataset305_Synapse2D        : 1.4279


### b) Compactness

In [596]:
from skimage.measure import label, regionprops

def compute_compactness_per_component(mask):
    labeled = label(mask)
    compactness_vals = []

    for region in regionprops(labeled):
        area = region.area
        if area == 0:
            continue

        perimeter = region.perimeter
        c = (perimeter ** 2) / (4 * np.pi * area)
        compactness_vals.append(c)

    return np.nanmean(compactness_vals)

In [597]:
dataset_ids = [300, 301, 302, 304, 305]

target_curv = {}
for dataset_id in dataset_ids:
    dataset_name = get_dataset_name(dataset_id)

    targets = glob(join(nnUNet_raw, dataset_name, "labelsTr", "*.png"))
    dataset_curv = []
    for target_path in tqdm(targets):
        target = np.array(Image.open(target_path))
        compactness = compute_compactness_per_component(target)
        dataset_curv.append(compactness)
        # plt.imshow(target)
        # plt.axis('off')
        # plt.show()
    target_curv[dataset_name] = np.nanmean(dataset_curv)

print("\nResults")
for dataset_name, curv in sorted(target_curv.items(), key=lambda x: x[1]):
    print(f"{dataset_name:<28}: {curv:.4f}")

  0%|          | 0/700 [00:00<?, ?it/s]/tmp/ipykernel_1077442/2632960438.py:16: RuntimeWarning: Mean of empty slice
  return np.nanmean(compactness_vals)
100%|██████████| 2211/2211 [00:05<00:00, 416.89it/s]


Results
Dataset300_isic2018         : 1.3687
Dataset304_BUSI             : 1.5093
Dataset305_Synapse2D        : 1.5350
Dataset301_busbra           : 1.5533
Dataset302_EchoNet-Dynamic  : 1.7448


### c) Curvature

In [594]:
import numpy as np
from skimage import measure
from scipy.ndimage import gaussian_filter1d

def compute_boundary_curvature_raggedness(
    mask, ignore_label=0, smooth_sigma=1.5, q=0.95, conn=2
    ):
    """
    mask: (H, W) label map
    Returns:
      - single scalar per image:
        mean over classes of (area-weighted q(|curvature|) over components of that class)
    """
    mask = np.asarray(mask)
    if mask.ndim != 2:
        raise ValueError("mask must be (H, W)")

    # connected components once
    cc = measure.label(mask != ignore_label, connectivity=conn)
    props = measure.regionprops(cc)

    per_class_vals = {}
    per_class_wts  = {}

    for r in props:
        comp_mask = (cc == r.label)

        # class id for this component
        cls = int(np.bincount(mask[comp_mask]).argmax())
        if cls == ignore_label:
            continue

        # contour
        contours = measure.find_contours(comp_mask.astype(np.uint8), level=0.5)
        if not contours:
            continue
        pts = max(contours, key=lambda a: a.shape[0])
        if pts.shape[0] < 5:
            continue

        y, x = pts[:, 0], pts[:, 1]
        x = gaussian_filter1d(x, sigma=smooth_sigma, mode="wrap")
        y = gaussian_filter1d(y, sigma=smooth_sigma, mode="wrap")

        dx, dy = np.gradient(x), np.gradient(y)
        ddx, ddy = np.gradient(dx), np.gradient(dy)

        denom = (dx*dx + dy*dy) ** 1.5 + 1e-6
        kappa = (dx * ddy - dy * ddx) / denom
        val = float(np.quantile(np.abs(kappa), q))

        per_class_vals.setdefault(cls, []).append(val)
        per_class_wts.setdefault(cls, []).append(r.area)

    # area-weighted raggedness per class, then average across classes
    class_scores = []
    for cls in per_class_vals:
        v = np.asarray(per_class_vals[cls])
        w = np.asarray(per_class_wts[cls])
        class_scores.append(np.average(v, weights=w))

    if not class_scores:
        return np.nan

    return float(np.mean(class_scores))


In [595]:
dataset_ids = [300, 301, 302, 304, 305]

target_curv = {}
for dataset_id in dataset_ids:
    dataset_name = get_dataset_name(dataset_id)

    targets = glob(join(nnUNet_raw, dataset_name, "labelsTr", "*.png"))
    dataset_curv = []
    for target_path in tqdm(targets):
        target = np.array(Image.open(target_path))
        compactness = compute_boundary_curvature_raggedness(target)
        dataset_curv.append(compactness)
        # plt.imshow(target)
        # plt.axis('off')
        # plt.show()
    target_curv[dataset_name] = np.nanmean(dataset_curv)

print("\nResults")
for dataset_name, curv in sorted(target_curv.items(), key=lambda x: x[1]):
    print(f"{dataset_name:<28}: {curv:.4f}")

100%|██████████| 2211/2211 [00:17<00:00, 125.50it/s]


Results
Dataset304_BUSI             : 0.1938
Dataset301_busbra           : 0.2232
Dataset305_Synapse2D        : 0.2238
Dataset302_EchoNet-Dynamic  : 0.3147
Dataset300_isic2018         : 0.4691


In [593]:
compactness

{1: 1.0664074325805002}

## Target - Image mutual information